In [0]:
df = spark.read.csv("/Volumes/insurance/dbo/insurance/medical_insurance.csv", header=True, inferSchema=True)

In [0]:
display(df.limit(10))

In [0]:
import uuid
from pyspark.sql import functions as F

In [0]:

df = df.withColumn("sl_no", F.expr("uuid()"))

display(df.limit(10))

In [0]:
# Reordering columns

first_col = ["sl_no"]

other_col = [i for i in df.columns if i!= "sl_no"]

col_reorder = first_col + other_col

df_reordered = df.select(col_reorder)

display(df_reordered.limit(10))

In [0]:
df_reordered.createOrReplaceTempView("df_reordered")

In [0]:
%%sql
-- To check if sl_no is unique using sql
select count(*) as total_count, count(distinct sl_no) as unique_count
from df_reordered

In [0]:
%%sql
-- Just to show t can be also done using spark sql window
-- function :)

with duplicate as (
    select *,
    row_number() over (
        partition by sl_no order by sl_no desc 
    ) as instances
    from df_reordered
)

select * from duplicate where instances>1

In [0]:
# Checking schema
df_reordered.printSchema()

In [0]:
# Change discount_eligibility to boolean

df_reordered = (
    df_reordered.withColumn("eligible_for_discount",
        F.expr("try_cast(discount_eligibility as boolean)")
    )
)

df_reordered.printSchema()


In [0]:
# Checking base stats

display(df_reordered.describe())

In [0]:
# check for nulls in all coloumn except sl_no
def null_logic(col_name):
    logic = F.count(
        F.when(
            F.col(col_name).isNull(), col_name)
            ).alias(col_name)
    return(logic)

null_count = df_reordered.select([null_logic(i) for i in col_reorder])

display(null_count)


In [0]:
# Cast premium col to 2 decimal places

df_reordered = (
    df_reordered.withColumn("premium",
    F.round("premium", 2))
)

display(df_reordered.limit(10))

In [0]:
# Including a random Yes or No Smoker column for depth

df_reordered = (
    df_reordered.withColumn(
        "smoker",
        F.when(F.rand(seed=42)<0.2, "yes")
        .otherwise("no")
    )
)

In [0]:
# Feature Engineering 1
# Binning of age groups

df_age_group= (
    df_reordered.withColumn("life_stage",
    F.when(((F.col("age") >= 18) & (F.col("age") <30)), "youth")
    .when(((F.col("age") >= 30) & (F.col("age") <60)), "adult")
    .otherwise("senior")
    )
)

In [0]:
# Feature Engineering 2
# Risk factor
df_risk = (
    df_age_group.withColumn("risk_factor",
    F.when((F.col("life_stage")== "youth") & (F.col("smoker") == "yes"), "Low")
    .when((F.col("life_stage")== "adult") & (F.col("smoker") == "yes"), "Medium")
    .when((F.col("life_stage")== "senior") & (F.col("smoker") == "yes"), "High")
    .otherwise("very_low")
    )
)

In [0]:
# Feature Engineering 3
# Obesity level
df_final = (
    df_risk.withColumn("obesity",
    F.when((F.col("bmi") < 18.5) , "underweight")
    .when((F.col("bmi") >= 18.5) & (F.col("bmi") < 25), "normal")
    .when((F.col("bmi") >= 25) & (F.col("bmi") < 30), "overweight")
    .otherwise("obese")
    )
)

In [0]:
df_final.printSchema()

In [0]:
df_final.write.format("delta").mode("overwrite").saveAsTable("insurance.dbo.insurance_gold")